# 009-R3baseline 完整v9基线最小 ICE forward（23支持分子/A16+B7，Colab T4）。按顺序跑，每格末尾有 EXPECT，对不上就停、贴回来。
只含23个支持分支研究分子（A16/B7，共523候选），4 NILU无窗跳过单列，不支持27不补ICE分，无保留集，不调参。truth仅本地离线打分用。YTCZZ（84候选，rank40）在包内，其forward结果即回答契约。预计实跑5-10分钟，保持页面在线。
3个文件（`v9r3baseline-bundle.tar.gz` + `gen.pt` + `inten.pt`）放 Drive 任意位置即可：第2格会自动搜，不用改路径。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
# EXPECT: Mounted at /content/drive
# 备选：不用 Drive 则跳过本格，把3个文件用左侧文件面板直接上传到 /content，第2格照样能找到。

In [ ]:
import os
SEARCH_DIRS = ["/content/drive/MyDrive/v9r3baseline-weights", "/content/drive/MyDrive", "/content"]
NEED = ["v9r3baseline-bundle.tar.gz", "gen.pt", "inten.pt"]
found = {}
for name in NEED:
    hit = next((d + "/" + name for d in SEARCH_DIRS if os.path.isfile(d + "/" + name)), None)
    found[name] = hit
    print(name, "->", hit if hit else "NOT FOUND")
print("searched:", SEARCH_DIRS)
missing = [k for k, v in found.items() if v is None]
assert not missing, "找不到: %s。请把这几个文件传到上面任一目录（Drive根目录最省事）后重跑本格。" % missing
os.environ["R3_TAR"] = found["v9r3baseline-bundle.tar.gz"]
os.environ["R3_GEN"] = found["gen.pt"]
os.environ["R3_INTEN"] = found["inten.pt"]
# EXPECT: 三行全部是具体路径（不是 NOT FOUND），无 AssertionError。后面的格都用这里定位到的路径，不再猜。

In [ ]:
import os, shutil
tar = os.environ["R3_TAR"]
shutil.rmtree("/content/v9r3baseline", ignore_errors=True)
# 注意：本包tar带顶层目录 v9r3baseline/，解压 -C /content 后即得 /content/v9r3baseline，不依赖 mkdir（修复flat-tar教训）
!tar xzf "{tar}" -C /content && test -f /content/v9r3baseline/inputs.json && ls /content/v9r3baseline
# EXPECT: 看到 inputs.json install.sh g1_worker.py MANIFEST.iceberg.json weights_reference.json README.md code/

In [ ]:
import os, hashlib, shutil
os.makedirs("/content/v9r3baseline/weights", exist_ok=True)
shutil.copy(os.environ["R3_GEN"], "/content/v9r3baseline/weights/gen.pt")
shutil.copy(os.environ["R3_INTEN"], "/content/v9r3baseline/weights/inten.pt")
!ls -l /content/v9r3baseline/weights/
EXP = {"gen.pt": ("deef0451c1870d2125653bb212c9d95c5b5d9bf80156dfa9b22fedda7db49bea", 14561244),
       "inten.pt": ("c49e330e049d3a8b0095026ed694fcf70045c24305354853732aaa5f89d55068", 13793037)}
for name, (sha, size) in EXP.items():
    p = "/content/v9r3baseline/weights/" + name
    assert os.path.getsize(p) == size, "%s bytes mismatch" % name
    h = hashlib.sha256(open(p, "rb").read()).hexdigest()
    assert h == sha, "%s sha mismatch: %s" % (name, h)
    print(name, "OK", size, sha[:12])
# runner/worker verbatim核对（必须与G1验证包一致）
import hashlib as _hl
ice_sha = _hl.sha256(open("/content/v9r3baseline/code/ice/ice_runner.py","rb").read()).hexdigest()
wkr_sha = _hl.sha256(open("/content/v9r3baseline/g1_worker.py","rb").read()).hexdigest()
assert ice_sha.startswith("dc0d5d1e"), "ice_runner SHA mismatch: %s" % ice_sha
assert wkr_sha.startswith("54b0751f"), "worker SHA mismatch: %s" % wkr_sha
print("ice_runner OK", ice_sha[:12])
print("worker OK", wkr_sha[:12])
# EXPECT: gen.pt OK / inten.pt OK / ice_runner OK dc0d5d1e / worker OK 54b0751f，无 AssertionError。mismatch 必须停，不要继续跑。

In [ ]:
import torch
assert torch.cuda.is_available(), "本格必须 T4 GPU 运行时：改 运行时→更改运行时类型→T4 后重跑"
print(torch.cuda.get_device_name(0))
# EXPECT: Tesla T4

In [ ]:
!cd /content/v9r3baseline && bash install.sh
# EXPECT: 末尾 install done. 约2-4分钟。

In [ ]:
!cd /content/v9r3baseline && PYTHONPATH=code/ice/shim:code/ice/src python3 g1_worker.py --pkg code/ice --model iceberg --inputs inputs.json --out /tmp/dry.json --k 100 --cemode split --dry-run --mol-limit 2
# EXPECT: DRY-RUN: 2 mols, 10 unique jobs，status ok-dry-run，无 Traceback。只验接线，不跑模型。（前2分子 CLSD4+UVYP6=10候选）

In [ ]:
!cd /content/v9r3baseline && mkdir -p scores cache work work/ice_ckpt_pkg/ckpt && ln -sf $(pwd)/weights/gen.pt work/ice_ckpt_pkg/ckpt/gen.pt && ln -sf $(pwd)/weights/inten.pt work/ice_ckpt_pkg/ckpt/inten.pt && ls -l work/ice_ckpt_pkg/ckpt/ && PYTHONPATH=code/ice/shim:code/ice/src python3 g1_worker.py --pkg code/ice --model iceberg --inputs inputs.json --out scores/v9r3baseline_ice_top100_split.json --k 100 --cemode split --device cuda --cache-dir cache --workdir work --ckpt-pkg work/ice_ckpt_pkg --any-rdkit --chunk-jobs 3000
# EXPECT: 末尾 status=ok（不是ok-dry-run）。约5-10分钟，meta 含 pred_per_s/max_mem_gb。全量23分子523 jobs纯预测约20s，大头是模型加载。

In [ ]:
# 可选 mean 对照（复用缓存，增量0，mean与split同523 jobs；不需要可跳过整格）
!cd /content/v9r3baseline && PYTHONPATH=code/ice/shim:code/ice/src python3 g1_worker.py --pkg code/ice --model iceberg --inputs inputs.json --out scores/v9r3baseline_ice_top100_mean.json --k 100 --cemode mean --device cuda --cache-dir cache --workdir work --ckpt-pkg work/ice_ckpt_pkg --any-rdkit
# EXPECT: status=ok。跳过不影响主结果回传。

In [ ]:
!cd /content/v9r3baseline && ls -lh scores/ && tar czf v9r3baseline-scores.tar.gz scores && ls -lh v9r3baseline-scores.tar.gz
# EXPECT: scores/ 下有 v9r3baseline_ice_top100_split.json(+.meta.json)；v9r3baseline-scores.tar.gz 若干MB。去左侧 Files 面板下载发回（+可选 cache/*.npz）。